<h1 styles="font-size=12rem; color:white;">EDA of Test Data</h1>

Creation of Fake data

In [ ]:
from faker import Faker
import pandas as pd

fake = Faker()

df = pd.DataFrame({
    "UIN": [i for i in range(1, 101)],
    "Name": [fake.name() for _ in range(100)],
    "Email": [fake.email() for _ in range(100)],
    "Phone": [fake.phone_number() for _ in range(100)]
})

df.to_excel("Test Data.xlsx", index=False)

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)

# Sample specialties & hobbies
specialties = ["Cardiology", "Diabetology", "Edocrinology"]
# Generate fake data
def generate_fake_data(n=5000):
    data = []
    for i in range(n):
        uin = f"UIN{i+1:04d}"
        mci_no = f"MCI{i+1:04d}"
        mci_state = fake.state()
        
        first_name = fake.first_name()
        middle_name = fake.first_name()
        last_name = fake.last_name()
        full_name = f"{first_name} {middle_name} {last_name}"
        
        specialty = random.choice(specialties)
        subspeciality = f"{specialty} - Advanced"
        qualification = random.choice(["MBBS", "MD", "DM", "MS", "PhD"])
        years_of_exp = random.randint(1, 40)
        
        work_phone_1 = fake.phone_number()
        work_phone_2 = fake.phone_number()
        mobile = fake.phone_number()
        whatsapp_phone = fake.phone_number()
        
        email1 = fake.email()
        email2 = fake.email()
        
        biography = fake.sentence(nb_words=12)
        institution = fake.company()
        department = random.choice(["Cardiology Dept", "Research Wing", "OPD"])
        position = random.choice(["Consultant", "Senior Doctor", "Resident"])
        
        clinic_address = fake.address().replace("\n", ", ")
        postal_code = fake.postcode()
        dr_city = fake.city()
        dr_state = fake.state()
        dr_country = fake.country()
        
        area_of_interest = random.choice(specialties)
        hobby = random.choice(hobbies) 
        urls = [fake.url() for _ in range(5)]
        
        ts = fake.date_time_between(start_date="-5y", end_date="now")
        processed_ts = ts
        inserted_ts = ts + timedelta(days=1)
        updated_ts = ts + timedelta(days=30)
        
        row = [
            uin, mci_no, mci_state, first_name, middle_name, last_name, full_name,
            specialty, specialty, subspeciality, qualification, years_of_exp,
            work_phone_1, work_phone_2, mobile, whatsapp_phone,
            email1, email2, biography, institution, department, position,
            clinic_address, postal_code, dr_city, dr_state, dr_country,
            area_of_interest, hobby,
            urls[0], urls[1], urls[2], urls[3], urls[4],
            processed_ts, inserted_ts, updated_ts,
            processed_ts.date(), inserted_ts.date(), updated_ts.date(),
            None if random.random() < 0.7 else fake.date_time_this_year()
        ]
        data.append(row)
    
    columns = [
        "UIN","MCI_no","MCI_state","first_name","middle_name","last_name","full_name",
        "uin_specialty","specialty","subspeciality","qualification","years_of_exp",
        "work_phone_1","work_phone_2","mobile","whatsapp_phone",
        "email_id_1","email_id_2","biography","primary_institution","primary_department","primary_position",
        "clinic_address","clinic_postal_code","dr_city","dr_state","dr_country",
        "area_of_interest","hobbies",
        "pi_source_url1","pi_source_url2","pi_source_url3","pi_source_url4","pi_source_url5",
        "pi_processed_timestamp","pi_inserted_timestamp","pi_updated_timestamp",
        "pi_processed_date","pi_inserted_date","pi_updated_date",
        "pi_delete_flag_timestamp"
    ]
    
    return pd.DataFrame(data, columns=columns)

# Generate 5000 fake rows
df = generate_fake_data(5000)

# Save to Excel
df.to_excel("synthetic_medical_data.xlsx", index=False)


Data Masking


In [38]:
import pandas as pd
import re

# Load the Excel file
file_path = "synthetic_medical_data.xlsx"
df = pd.read_excel(file_path)

# --- Masking helper functions ---
def mask_id(val):
    if pd.isna(val): return val
    return str(val)[:3] + "*" * (len(str(val)) - 3)

def mask_name(name):
    if pd.isna(name): return name
    return name[0] + "*" * (len(name) - 1)

def mask_phone(phone):
    if pd.isna(phone): return phone
    s = re.sub(r"\D", "", str(phone))  # digits only
    return "*" * (len(s) - 2) + s[-2:] if len(s) >= 2 else "*" * len(s)

def mask_email(email):
    if pd.isna(email): return email
    try:
        user, domain = email.split("@")
        return user[0] + "***@" + domain
    except:
        return "***masked***"

def mask_text(val):
    if pd.isna(val): return val
    return val[:5] + "..." if len(str(val)) > 5 else "***"

def mask_url(url):
    if pd.isna(url): return url
    return "https://***masked***.com"

def mask_date(date_val):
    if pd.isna(date_val): return date_val
    try:
        return str(date_val)[:4] + "-**-**"  # Keep year only
    except:
        return "***"

# --- Apply masking ---
df["UIN"] = df["UIN"].apply(mask_id)
df["MCI_no"] = df["MCI_no"].apply(mask_id)

for col in ["first_name","middle_name","last_name","full_name"]:
    df[col] = df[col].apply(mask_name)

for col in ["work_phone_1","work_phone_2","mobile","whatsapp_phone"]:
    df[col] = df[col].apply(mask_phone)

for col in ["email_id_1","email_id_2"]:
    df[col] = df[col].apply(mask_email)

for col in ["biography","clinic_address","primary_institution","primary_department","primary_position"]:
    df[col] = df[col].apply(mask_text)

for col in ["pi_source_url1","pi_source_url2","pi_source_url3","pi_source_url4","pi_source_url5"]:
    df[col] = df[col].apply(mask_url)

for col in ["pi_processed_timestamp","pi_inserted_timestamp","pi_updated_timestamp",
            "pi_processed_date","pi_inserted_date","pi_updated_date","pi_delete_flag_timestamp"]:
    df[col] = df[col].apply(mask_date)

# Save masked version
output_path = "masked_synthetic_medical_data.xlsx"
df.to_excel(output_path, index=False)

print(f"Masked file saved as: {output_path}")


Masked file saved as: masked_synthetic_medical_data.xlsx


#There is nothing more convoluted than this.